# 05 · Comparing models on your own task

Tutorial section 11 ([`TUTORIAL.md`](../TUTORIAL.md#11-comparing-models-on-your-own-task)). Offline, the answers are scripted so the comparison has something to show; with `LIVE=1` the same loop runs against the three Bedrock models. There is no `temperature=0`: Claude 5.5 models reject non-default sampling parameters.

All models are reached through AWS Bedrock. Run the setup cell first: it loads `.env.local` next to this notebook (AWS profile, region, model choice, `LIVE` switch) and prints those settings. Credentials come from your AWS SSO profile, never from the notebook or the file. With `LIVE` unset or `0`, every call returns a mock response: no AWS access, no cost. See `00_credentials.ipynb` to set up.

`FAST`, `SMART` and `ALT` are the three Bedrock models from `.env.local` (defaults: Claude Haiku 5.5, Claude Opus 5.5, Amazon Nova 2 Lite).

In [1]:
from _env import ALT, FAST, LIVE, MODELS, SMART, mock, status  # loads .env.local; import before litellm

import litellm

litellm.suppress_debug_info = True
status()

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


In [2]:
import asyncio
import time

import pandas as pd

litellm.drop_params = True

CASES = [
    ("What is 17 * 23? Reply with the number only.", "391"),
    ("Capital of Canada? One word.", "Ottawa"),
    ("Is 91 prime? Reply yes or no.", "no"),
    ("Reverse the string 'litellm'. Reply with the result only.", "mlletil"),
]
FAKE = dict(zip(MODELS, [
    ["391", "Ottawa", "yes", "mlletil"],
    ["391", "Ottawa", "No", "mlletil"],
    ["391", "Toronto", "no", "mletlil"],
], strict=True))


async def run_case(model, i, prompt):
    fake = {} if LIVE else {"mock_response": FAKE[model][i], "mock_delay": 0.05 * (i + 1)}
    start = time.perf_counter()
    resp = await litellm.acompletion(model=model, messages=[{"role": "user", "content": prompt}],
                                     max_tokens=1024, **fake)
    return ((resp.choices[0].message.content or "").strip(), litellm.completion_cost(resp),
            time.perf_counter() - start)


rows = []
for model in MODELS:
    results = await asyncio.gather(*(run_case(model, i, p) for i, (p, _) in enumerate(CASES)))
    for (prompt, want), (out, cost, secs) in zip(CASES, results, strict=True):
        rows.append({"model": model, "prompt": prompt, "want": want, "got": out,
                     "correct": out.lower().strip(".") == want.lower(), "cost": cost, "secs": secs})

df = pd.DataFrame(rows)
df

,model,prompt,want,got,correct,cost,secs
0,bedrock/us.anthropic.claude-haiku-5-5,What is 17 * 23? Reply with the number only.,391,391,True,0.000004,3.409036
1,bedrock/us.anthropic.claude-haiku-5-5,Capital of Canada? One word.,Ottawa,Ottawa,True,0.000006,3.418266
2,bedrock/us.anthropic.claude-haiku-5-5,Is 91 prime? Reply yes or no.,no,No,True,0.000004,3.107602
3,bedrock/us.anthropic.claude-haiku-5-5,Reverse the string 'litellm'. Reply with the r...,mlletil,mtellil,False,0.000006,3.232747
4,bedrock/us.anthropic.claude-opus-5-5,What is 17 * 23? Reply with the number only.,391,391,True,0.003102,2.586705
5,bedrock/us.anthropic.claude-opus-5-5,Capital of Canada? One word.,Ottawa,Ottawa,True,0.000238,1.191443
6,bedrock/us.anthropic.claude-opus-5-5,Is 91 prime? Reply yes or no.,no,No. 91 = 7 × 13.,False,0.000810,1.393737
7,bedrock/us.anthropic.claude-opus-5-5,Reverse the string 'litellm'. Reply with the r...,mlletil,mlletil,True,0.001338,1.577953
8,bedrock/us.amazon.nova-2-lite-v1:0,What is 17 * 23? Reply with the number only.,391,391,True,0.000025,0.761747
9,bedrock/us.amazon.nova-2-lite-v1:0,Capital of Canada? One word.,Ottawa,The capital of Canada is **Ottawa**.,False,0.000048,1.092455


Per-model summary.

In [3]:
df.groupby("model").agg(accuracy=("correct", "mean"), cost_usd=("cost", "sum"),
                        p50_secs=("secs", "median")).round(6)

,accuracy,cost_usd,p50_secs
model,,,
bedrock/us.amazon.nova-2-lite-v1:0,0.25,0.000370,1.059684
bedrock/us.anthropic.claude-haiku-5-5,0.75,0.000021,3.320891
bedrock/us.anthropic.claude-opus-5-5,0.75,0.005487,1.485845
